# Posture Loss and Contact-Point Consistency

This study connects a hitter's post-plant trunk posture to the consistency of the hand position and HitTrax point of impact at contact.

## Research questions

1. Is greater post-plant posture loss associated with more variable hand position at contact?
2. Is the timing of peak post-plant trunk tilt associated with more variable HitTrax point-of-impact depth?

This is an observational association study. It does not label cut balls, measure true smash factor, or establish that posture loss causes a particular batted-ball outcome. Exit velocity is intentionally not required for the primary analysis.

## Analysis unit

The notebook calculates swing-level features first, then aggregates them within athlete-session. Only athlete-sessions with at least five complete swings are retained. Associations use Spearman correlations with athlete-level bootstrap confidence intervals so repeated swings and repeated sessions do not receive inappropriate independence.

In [ ]:
from pathlib import Path
import sys
import hashlib
import zipfile
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_SEED = 20260916
MIN_SWINGS_PER_SESSION = 5
N_BOOTSTRAPS = 5000

# Run this notebook from inside the baseball-biomechanics-analysis repository.
PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "obp_utils.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook from inside "
        "the baseball-biomechanics-analysis repository."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "hitting"
SELECTED_SWING = None

In [ ]:
LANDMARKS_URL = (
    "https://github.com/drivelineresearch/openbiomechanics/releases/download/"
    "dataset-v1/hitting_landmarks.zip"
)
LANDMARKS_SHA256 = (
    "9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3"
)
METADATA_URL = (
    "https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/"
    "dataset-v1/baseball_hitting/data/metadata.csv"
)
POI_URL = (
    "https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/"
    "dataset-v1/baseball_hitting/data/poi/poi_metrics.csv"
)
HITTRAX_URL = (
    "https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/"
    "dataset-v1/baseball_hitting/data/poi/hittrax.csv"
)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def first_existing(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def download_csv_if_missing(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {path.name}...")
        urlretrieve(url, path)
    return path


def download_landmarks_if_missing(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        return path

    archive = path.parent / "hitting_landmarks.zip"
    if not archive.exists():
        print("Downloading official OBP hitting landmarks...")
        urlretrieve(LANDMARKS_URL, archive)

    actual_hash = sha256_file(archive)
    if actual_hash != LANDMARKS_SHA256:
        raise ValueError(
            "Checksum mismatch for hitting_landmarks.zip. "
            f"Expected {LANDMARKS_SHA256}, got {actual_hash}."
        )

    with zipfile.ZipFile(archive) as compressed:
        members = [
            member for member in compressed.namelist()
            if member.endswith("landmarks.csv")
        ]
        if len(members) != 1:
            raise ValueError(
                "Expected one landmarks.csv file in the official archive, "
                f"found {len(members)}."
            )
        with compressed.open(members[0]) as source, path.open("wb") as target:
            target.write(source.read())

    return path


def prepare_hitting_inputs():
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    metadata_path = first_existing(
        [
            RAW_DIR / "metadata.csv",
            Path.cwd() / "obp_hinge_data" / "metadata.csv",
            PROJECT_ROOT / "obp_hinge_data" / "metadata.csv",
            PROJECT_ROOT / "openbiomechanics" / "baseball_hitting"
            / "data" / "metadata.csv",
        ]
    )
    if metadata_path is None:
        metadata_path = download_csv_if_missing(
            METADATA_URL, RAW_DIR / "metadata.csv"
        )

    landmarks_path = first_existing(
        [
            RAW_DIR / "landmarks.csv",
            Path.cwd() / "obp_hinge_data" / "landmarks.csv",
            PROJECT_ROOT / "obp_hinge_data" / "landmarks.csv",
            PROJECT_ROOT / "openbiomechanics" / "baseball_hitting"
            / "data" / "full_sig" / "landmarks.csv",
        ]
    )
    if landmarks_path is None:
        landmarks_path = download_landmarks_if_missing(
            RAW_DIR / "landmarks.csv"
        )

    poi_path = first_existing(
        [
            RAW_DIR / "poi_metrics.csv",
            PROJECT_ROOT / "openbiomechanics" / "baseball_hitting"
            / "data" / "poi" / "poi_metrics.csv",
        ]
    )
    if poi_path is None:
        poi_path = download_csv_if_missing(
            POI_URL, RAW_DIR / "poi_metrics.csv"
        )

    hittrax_path = first_existing(
        [
            RAW_DIR / "hittrax.csv",
            PROJECT_ROOT / "openbiomechanics" / "baseball_hitting"
            / "data" / "poi" / "hittrax.csv",
        ]
    )
    if hittrax_path is None:
        hittrax_path = download_csv_if_missing(
            HITTRAX_URL, RAW_DIR / "hittrax.csv"
        )

    return metadata_path, landmarks_path, poi_path, hittrax_path


metadata_path, landmarks_path, poi_path, hittrax_path = prepare_hitting_inputs()

print("Metadata: ", metadata_path)
print("Landmarks:", landmarks_path)
print("POI:      ", poi_path)
print("HitTrax:  ", hittrax_path)

In [ ]:
LANDMARK_BASES = [
    "left_hip", "right_hip", "lsjc", "rsjc",
    "lhjc", "rhjc",
]
COORD_COLUMNS = [
    f"{name}_{axis}"
    for name in LANDMARK_BASES
    for axis in "xyz"
]
EVENT_COLUMNS = ["fp_100_time", "contact_time"]

landmarks = pd.read_csv(
    landmarks_path,
    usecols=["session_swing", "time", *COORD_COLUMNS, *EVENT_COLUMNS],
    dtype={"session_swing": str},
)
metadata = pd.read_csv(
    metadata_path,
    dtype={"session_swing": str, "user": str, "session": str},
)
poi = pd.read_csv(
    poi_path,
    usecols=[
        "session_swing",
        "exit_velo_mph_x",
        "bat_speed_mph_contact_x",
        "attack_angle_contact_x",
    ],
    dtype={"session_swing": str},
)
hittrax = pd.read_csv(
    hittrax_path,
    usecols=["session_swing", "poi_x", "poi_y", "poi_z"],
    dtype={"session_swing": str},
)

for frame in [landmarks, metadata, poi, hittrax]:
    frame["session_swing"] = frame["session_swing"].astype(str)

for name, frame in [
    ("metadata", metadata),
    ("POI", poi),
    ("HitTrax", hittrax),
]:
    if frame["session_swing"].duplicated().any():
        raise ValueError(f"{name} contains duplicate session_swing keys.")

metadata_columns = [
    "session_swing",
    "user",
    "session",
    "hitter_side",
    "athlete_age",
    "highest_playing_level",
]
kinematics = landmarks.merge(
    metadata[metadata_columns].drop_duplicates("session_swing"),
    on="session_swing",
    how="left",
    validate="many_to_one",
)

if kinematics["user"].isna().any():
    raise ValueError("Some landmarks rows did not match metadata.")

print(f"Landmark rows: {len(kinematics):,}")
print(f"Swing trials:  {kinematics['session_swing'].nunique():,}")
print(f"Athletes:      {kinematics['user'].nunique():,}")
print(
    "Median landmark sampling interval: "
    f"{kinematics.groupby('session_swing')['time'].diff().median():.4f} seconds"
)

## Feature definitions

The coordinate system is kept in the original OpenBiomechanics convention. For posture, the trunk vector is the midpoint of the hips to the midpoint of the shoulders. Trunk tilt is the absolute angle between that vector and vertical in the sagittal x-z plane.

- Posture loss at contact = trunk tilt at contact minus trunk tilt at foot plant.
- Maximum post-plant posture loss = the largest trunk tilt between foot plant and contact minus trunk tilt at foot plant.
- Peak-tilt timing = milliseconds from foot plant to the frame with maximum trunk tilt.
- Hand position = the midpoint of the left- and right-hand joint centers at contact, expressed relative to the hip midpoint and normalized by shoulder width.
- Hand-contact variability = the three-dimensional root-sum-square of the within-session standard deviations of the normalized hand coordinates.
- HitTrax point-of-impact depth variability = the within-session standard deviation of poi_z, reported in the dataset's coordinate units.

The two hand joint centers are used as a reproducible hand-position proxy. HitTrax poi_z is kept as a separate point-of-impact measure and is not treated as hand depth.

In [ ]:
def make_vector(frame, start, end):
    return np.column_stack(
        [
            frame[f"{end}_{axis}"].to_numpy(dtype=float)
            - frame[f"{start}_{axis}"].to_numpy(dtype=float)
            for axis in "xyz"
        ]
    )


def angle_from_vertical_xz(vector):
    denominator = np.linalg.norm(vector, axis=1)
    if np.any(denominator == 0):
        raise ValueError("A zero-length trunk vector was found.")
    return np.degrees(
        np.arctan2(np.abs(vector[:, 0]), np.abs(vector[:, 2]))
    )


hip_mid = np.column_stack(
    [
        (
            kinematics[f"left_hip_{axis}"].to_numpy(dtype=float)
            + kinematics[f"right_hip_{axis}"].to_numpy(dtype=float)
        ) / 2
        for axis in "xyz"
    ]
)
shoulder_mid = np.column_stack(
    [
        (
            kinematics[f"lsjc_{axis}"].to_numpy(dtype=float)
            + kinematics[f"rsjc_{axis}"].to_numpy(dtype=float)
        ) / 2
        for axis in "xyz"
    ]
)
hand_mid = np.column_stack(
    [
        (
            kinematics[f"lhjc_{axis}"].to_numpy(dtype=float)
            + kinematics[f"rhjc_{axis}"].to_numpy(dtype=float)
        ) / 2
        for axis in "xyz"
    ]
)

trunk_vector = shoulder_mid - hip_mid
shoulder_width = np.linalg.norm(
    np.column_stack(
        [
            kinematics[f"lsjc_{axis}"].to_numpy(dtype=float)
            - kinematics[f"rsjc_{axis}"].to_numpy(dtype=float)
            for axis in "xyz"
        ]
    ),
    axis=1,
)
if np.any(shoulder_width <= 0):
    raise ValueError("Non-positive shoulder width encountered.")

kinematics["trunk_tilt_deg"] = angle_from_vertical_xz(trunk_vector)

for column_index, axis in enumerate("xyz"):
    kinematics[f"hand_mid_{axis}"] = hand_mid[:, column_index]
    kinematics[f"hip_mid_{axis}"] = hip_mid[:, column_index]
    kinematics[f"shoulder_mid_{axis}"] = shoulder_mid[:, column_index]
    kinematics[f"hand_rel_{axis}_norm"] = (
        hand_mid[:, column_index] - hip_mid[:, column_index]
    ) / shoulder_width

In [ ]:
def nearest_frame(group, target_time):
    if pd.isna(target_time):
        return None
    distances = np.abs(group["time"].to_numpy(dtype=float) - float(target_time))
    return group.iloc[int(distances.argmin())]


def extract_swing_features(group):
    group = group.sort_values("time").reset_index(drop=True)
    plant_time = group["fp_100_time"].iloc[0]
    contact_time = group["contact_time"].iloc[0]

    if pd.isna(plant_time) or pd.isna(contact_time):
        return None
    if contact_time <= plant_time:
        return None

    window = group[group["time"].between(plant_time, contact_time)].copy()
    if len(window) < 3:
        return None

    plant = nearest_frame(group, plant_time)
    contact = nearest_frame(group, contact_time)
    peak_position = int(window["trunk_tilt_deg"].to_numpy().argmax())
    peak = window.iloc[peak_position]

    peak_tilt = float(peak["trunk_tilt_deg"])
    contact_tilt = float(contact["trunk_tilt_deg"])
    plant_tilt = float(plant["trunk_tilt_deg"])
    recovery_deg = peak_tilt - contact_tilt
    uprighting_started = bool(
        (peak["time"] < contact["time"]) and (recovery_deg > 0)
    )

    return {
        "session_swing": str(group["session_swing"].iloc[0]),
        "user": str(group["user"].iloc[0]),
        "session": str(group["session"].iloc[0]),
        "hitter_side": group["hitter_side"].iloc[0],
        "plant_time": float(plant["time"]),
        "contact_time": float(contact["time"]),
        "plant_tilt_deg": plant_tilt,
        "contact_tilt_deg": contact_tilt,
        "posture_loss_at_contact_deg": contact_tilt - plant_tilt,
        "max_postplant_posture_loss_deg": peak_tilt - plant_tilt,
        "peak_tilt_time_ms": 1000
        * (float(peak["time"]) - float(plant["time"])),
        "recovery_deg": recovery_deg,
        "uprighting_started_before_contact": uprighting_started,
        "uprighting_duration_ms": (
            1000 * (float(contact["time"]) - float(peak["time"]))
            if uprighting_started
            else 0.0
        ),
        "hand_contact_x_norm": float(contact["hand_rel_x_norm"]),
        "hand_contact_y_norm": float(contact["hand_rel_y_norm"]),
        "hand_contact_z_norm": float(contact["hand_rel_z_norm"]),
        "hand_contact_x": float(contact["hand_mid_x"]),
        "hand_contact_y": float(contact["hand_mid_y"]),
        "hand_contact_z": float(contact["hand_mid_z"]),
        "frames_in_postplant_window": len(window),
    }


swing_features = pd.DataFrame(
    [
        record
        for _, group in kinematics.groupby("session_swing", sort=False)
        if (record := extract_swing_features(group)) is not None
    ]
)

if swing_features.empty:
    raise ValueError(
        "No complete swings were found. Check the event columns and data paths."
    )

print(f"Complete swing feature rows: {len(swing_features):,}")
print(f"Athletes represented:        {swing_features['user'].nunique():,}")
display(swing_features.head())

In [ ]:
swing_data = swing_features.merge(
    poi,
    on="session_swing",
    how="left",
    validate="one_to_one",
).merge(
    hittrax,
    on="session_swing",
    how="left",
    validate="one_to_one",
)

print(
    "Complete swing rows with POI exit velocity: "
    f"{swing_data['exit_velo_mph_x'].notna().sum():,}"
)
print(
    "Complete swing rows with HitTrax point-of-impact depth: "
    f"{swing_data['poi_z'].notna().sum():,}"
)


def summarize_session(group):
    group = group.sort_values("session_swing")
    result = {
        "user": str(group["user"].iloc[0]),
        "session": str(group["session"].iloc[0]),
        "hitter_side": group["hitter_side"].iloc[0],
        "swings": len(group),
        "posture_loss_mean_deg": group["posture_loss_at_contact_deg"].mean(),
        "max_postplant_posture_loss_mean_deg": (
            group["max_postplant_posture_loss_deg"].mean()
        ),
        "peak_tilt_time_mean_ms": group["peak_tilt_time_ms"].mean(),
        "uprighting_started_pct": (
            100 * group["uprighting_started_before_contact"].mean()
        ),
        "uprighting_duration_mean_ms": group["uprighting_duration_ms"].mean(),
    }

    hand_sd = {}
    for axis in "xyz":
        hand_sd[axis] = group[f"hand_contact_{axis}_norm"].std(ddof=1)
        result[f"hand_contact_{axis}_sd_norm"] = hand_sd[axis]

    result["hand_contact_spread_3d_norm"] = np.sqrt(
        sum(value**2 for value in hand_sd.values())
    )

    depth = group["poi_z"].dropna()
    result["hittrax_depth_swings"] = len(depth)
    result["hittrax_poi_depth_mean"] = depth.mean() if len(depth) else np.nan
    result["hittrax_poi_depth_sd"] = (
        depth.std(ddof=1) if len(depth) >= 2 else np.nan
    )

    return result


session_summary = pd.DataFrame(
    [
        summarize_session(group)
        for _, group in swing_data.groupby(["user", "session"], sort=False)
        if len(group) >= MIN_SWINGS_PER_SESSION
    ]
)

if session_summary.empty:
    raise ValueError(
        "No athlete-sessions met the five-complete-swing minimum."
    )

session_summary["user_session"] = (
    session_summary["user"] + "_" + session_summary["session"]
)

print(
    f"Retained athlete-sessions: {len(session_summary):,} "
    f"from {session_summary['user'].nunique():,} athletes"
)
display(
    session_summary[
        [
            "user", "session", "swings",
            "posture_loss_mean_deg",
            "max_postplant_posture_loss_mean_deg",
            "peak_tilt_time_mean_ms",
            "hand_contact_spread_3d_norm",
            "hittrax_depth_swings",
            "hittrax_poi_depth_sd",
        ]
    ].head(10).round(3)
)

## Session-level association analysis

The primary hand-position analysis uses all retained athlete-sessions. The HitTrax depth analysis uses only sessions with at least five non-missing poi_z observations.

The bootstrap resamples athletes, not individual swings. This preserves the dependence among multiple sessions from the same athlete and produces uncertainty intervals that are more conservative than treating every session as independent.

In [ ]:
def athlete_bootstrap_correlation(
    data,
    x_column,
    y_column,
    n_boot=N_BOOTSTRAPS,
    seed=RANDOM_SEED,
):
    valid = data[[x_column, y_column, "user"]].dropna().copy()

    if len(valid) < 3 or valid[x_column].nunique() < 2 or valid[y_column].nunique() < 2:
        return {
            "sessions": len(valid),
            "athletes": valid["user"].nunique(),
            "spearman_rho": np.nan,
            "p_value": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    rho, p_value = stats.spearmanr(valid[x_column], valid[y_column])
    athlete_ids = valid["user"].unique()
    rng = np.random.default_rng(seed)
    bootstrap_values = []

    for _ in range(n_boot):
        sampled_ids = rng.choice(
            athlete_ids,
            size=len(athlete_ids),
            replace=True,
        )
        sampled_groups = [
            valid[valid["user"].eq(athlete_id)]
            for athlete_id in sampled_ids
        ]
        draw = pd.concat(sampled_groups, ignore_index=True)

        if draw[x_column].nunique() < 2 or draw[y_column].nunique() < 2:
            continue

        draw_rho, _ = stats.spearmanr(draw[x_column], draw[y_column])
        if pd.notna(draw_rho):
            bootstrap_values.append(draw_rho)

    if bootstrap_values:
        ci_low, ci_high = np.quantile(bootstrap_values, [0.025, 0.975])
    else:
        ci_low, ci_high = np.nan, np.nan

    return {
        "sessions": len(valid),
        "athletes": valid["user"].nunique(),
        "spearman_rho": rho,
        "p_value": p_value,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }

In [ ]:
relationships = [
    {
        "relationship": "Posture loss → hand-contact variability",
        "data": session_summary,
        "x": "posture_loss_mean_deg",
        "y": "hand_contact_spread_3d_norm",
    },
    {
        "relationship": "Maximum posture loss → hand-contact variability",
        "data": session_summary,
        "x": "max_postplant_posture_loss_mean_deg",
        "y": "hand_contact_spread_3d_norm",
    },
    {
        "relationship": "Peak-tilt timing → HitTrax depth variability",
        "data": session_summary[
            session_summary["hittrax_depth_swings"] >= MIN_SWINGS_PER_SESSION
        ],
        "x": "peak_tilt_time_mean_ms",
        "y": "hittrax_poi_depth_sd",
    },
]

relationship_rows = []
for relationship in relationships:
    result = athlete_bootstrap_correlation(
        relationship["data"],
        relationship["x"],
        relationship["y"],
    )
    relationship_rows.append(
        {
            "relationship": relationship["relationship"],
            **result,
        }
    )

relationship_results = pd.DataFrame(relationship_rows)
display(relationship_results.round(3))

In [ ]:
def add_relationship_plot(
    ax,
    data,
    x_column,
    y_column,
    x_label,
    y_label,
    title,
):
    valid = data[[x_column, y_column]].dropna()

    if len(valid) < 3:
        ax.text(
            0.5,
            0.5,
            "Not enough complete observations",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title(title)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        return

    ax.scatter(
        valid[x_column],
        valid[y_column],
        color="#7A0019",
        alpha=0.78,
        edgecolor="white",
        linewidth=0.6,
        s=54,
    )

    if valid[x_column].nunique() >= 2:
        slope, intercept = np.polyfit(
            valid[x_column],
            valid[y_column],
            1,
        )
        line_x = np.linspace(
            valid[x_column].min(),
            valid[x_column].max(),
            100,
        )
        ax.plot(
            line_x,
            slope * line_x + intercept,
            color="#555555",
            linestyle="--",
            linewidth=1.5,
        )

    rho, _ = stats.spearmanr(valid[x_column], valid[y_column])
    ax.text(
        0.03,
        0.95,
        f"Spearman rho = {rho:.2f}\nn = {len(valid)}",
        transform=ax.transAxes,
        va="top",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)


fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
add_relationship_plot(
    axes[0],
    session_summary,
    "posture_loss_mean_deg",
    "hand_contact_spread_3d_norm",
    "Mean posture loss at contact (degrees)",
    "Hand-contact variability (normalized 3D SD)",
    "Posture loss vs. hand-contact consistency",
)
add_relationship_plot(
    axes[1],
    session_summary[
        session_summary["hittrax_depth_swings"] >= MIN_SWINGS_PER_SESSION
    ],
    "peak_tilt_time_mean_ms",
    "hittrax_poi_depth_sd",
    "Mean time to peak trunk tilt (ms after plant)",
    "HitTrax poi_z variability (dataset units)",
    "Peak-tilt timing vs. point-of-impact depth consistency",
)
fig.suptitle("Posture and contact-point relationships", fontsize=15)
fig.tight_layout()

figure_path = (
    PROJECT_ROOT
    / "figures"
    / "generated"
    / "04_posture_contact_consistency.png"
)
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved figure to {figure_path}")

In [ ]:
def plot_selected_swing(session_swing):
    selected = kinematics[
        kinematics["session_swing"].eq(str(session_swing))
    ].sort_values("time")

    if selected.empty:
        raise KeyError(f"Swing {session_swing!r} was not found.")

    plant_time = float(selected["fp_100_time"].iloc[0])
    contact_time = float(selected["contact_time"].iloc[0])
    window = selected[
        selected["time"].between(plant_time - 0.10, contact_time + 0.05)
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

    axes[0].plot(
        window["time"],
        window["trunk_tilt_deg"],
        color="#7A0019",
        linewidth=2.4,
    )
    axes[0].axvline(
        plant_time,
        color="#1565C0",
        linestyle="--",
        label="Foot plant",
    )
    axes[0].axvline(
        contact_time,
        color="#D95F02",
        linestyle="--",
        label="Contact",
    )
    axes[0].set_title(f"Trunk tilt — swing {session_swing}")
    axes[0].set_xlabel("Capture time (seconds)")
    axes[0].set_ylabel("Absolute trunk tilt from vertical (degrees)")
    axes[0].legend()

    axes[1].plot(
        window["time"],
        window["hand_rel_x_norm"],
        label="Hand position x",
        linewidth=2,
    )
    axes[1].plot(
        window["time"],
        window["hand_rel_y_norm"],
        label="Hand position y",
        linewidth=2,
    )
    axes[1].plot(
        window["time"],
        window["hand_rel_z_norm"],
        label="Hand position z",
        linewidth=2,
    )
    axes[1].axvline(plant_time, color="#1565C0", linestyle="--")
    axes[1].axvline(contact_time, color="#D95F02", linestyle="--")
    axes[1].set_title("Normalized hand position")
    axes[1].set_xlabel("Capture time (seconds)")
    axes[1].set_ylabel("Position relative to hip midpoint / shoulder width")
    axes[1].legend()

    fig.tight_layout()
    plt.show()


if SELECTED_SWING is None:
    SELECTED_SWING = str(
        swing_features.sort_values("session_swing")["session_swing"].iloc[0]
    )

plot_selected_swing(SELECTED_SWING)

## Interpretation guide

A positive relationship in the first analysis would mean that athlete-sessions with greater average posture loss also showed more variable hand locations at contact. A positive relationship in the second analysis would mean that later peak-tilt timing coincided with more variable HitTrax point-of-impact depth.

These are sample-level associations, not mechanical prescriptions for an individual hitter. A null relationship would also be useful: it would suggest that the proposed posture metric does not explain much of the observed contact variation in this controlled sample.

## Limitations

- The public data do not directly label cut balls, square-up quality, or true smash factor.
- The hand midpoint is a marker-derived proxy for hand position, not a direct bat-ball contact measurement.
- HitTrax poi_z is a separate point-of-impact coordinate and is not treated as hand depth.
- The swings were collected from a pitching machine at approximately 65 mph from approximately 40 feet, so the sample is not equivalent to game swings.
- The analysis is observational and cross-sectional. It identifies associations rather than causation.
- The OpenBiomechanics data and documentation have separate licensing terms. Review the current data license before publishing or using the analysis professionally.

## Sources

- Driveline Baseball OpenBiomechanics Project
- OpenBiomechanics baseball hitting documentation
- OpenBiomechanics data dictionary and dataset-v1 release files